In [ ]:
!pip install lingua-language-detector

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import random
import os
import nltk
from lingua import Language, LanguageDetectorBuilder
import seaborn as sns
import matplotlib.pyplot as plt
import time
import gc
import ast
import json
from collections import Counter
from pathlib import Path
import math
import warnings
from tqdm.auto import tqdm
import itertools

from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,f1_score
)
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

In [ ]:
import torch
#from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging as hf_logging
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed as hf_set_seed,
)

if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
import sys, platform, subprocess, importlib.metadata as md

print("Python:", sys.version.split()[0], "|", platform.platform())

# GPU and CUDA
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
try:
    import torch
    print("torch:", torch.__version__, "| CUDA:", torch.version.cuda,
          "| cuDNN:", torch.backends.cudnn.version(),
          "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
except Exception as e:
    print("torch not available:", e)

# Library versions (edit the list to match what your notebook imports)
pkgs = ["transformers", "tokenizers", "accelerate", "datasets", "evaluate",
        "scikit-learn", "numpy", "pandas", "scipy", "nltk",
        "lingua-language-detector"]
for p in pkgs:
    try:
        print(f"{p}: {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p}: not installed")

# NLTK words corpus size (the corpus has no version number, so record its size)
try:
    import nltk
    nltk.download("words", quiet=True)
    from nltk.corpus import words
    print("NLTK words corpus entries:", len(words.words()))
except Exception as e:
    print("nltk words:", e)

# Save the full environment so it can be attached to the supplementary file
open("requirements_colab.txt", "w").write(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print("Saved requirements_colab.txt")

In [ ]:
RANDOM_SEED = 42

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)  # menyamakan seed init model, data sampler, dsb di HF

set_all_seeds(RANDOM_SEED)

In [ ]:
df = pd.read_csv('code_switched_en_id_preprocessed_filtered.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.sentiment.value_counts()

In [ ]:
df.head()

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
df_shuffled = shuffle(df, random_state=RANDOM_SEED)

In [ ]:
df_shuffled.head()

In [ ]:
def check_error_from_predictions(predictions):
  # Map predictions back to the structured evaluation dataframe
  results_df = X_test.copy().reset_index(drop=True)
  results_df['sentiment'] = y_test.reset_index(drop=True)
  results_df['predicted_sentiment'] = predictions
  # Identify exactly where predictions diverged from true sentiment
  results_df['is_error'] = results_df['sentiment'] != results_df['predicted_sentiment']


  # --- 4. EXPORT TARGETED ERROR ROWS FOR MANUAL INSPECTION ---
  # Captures sample text sequences to diagnose failure modes (e.g., tokenization flaws)
  error_samples = results_df[results_df['is_error'] == True][['text', 'interleaved', 'adjacent', 'concatenated', 'sentiment', 'predicted_sentiment', 'uuid']]

  return error_samples

In [ ]:
# #Untuk membantu agar tidak crash saat training model nya
# # 3. CRITICAL: Clear Pandas from RAM to prevent system crash
# del X_train
# del X_train_val
# gc.collect()

#TRANSFORMER MODELS

In [ ]:
# Reduce batch size to save memory
MAX_SAMPLES= 1000
BATCH_SIZE = 32
MAX_SEQ_LEN = 400
EPOCHS = 4
LEARNING_RATE = 2e-5

In [ ]:
# Seed untuk multi-seed run (inisialisasi bobot, shuffling batch, dsb).
# Tiap kombinasi model x representasi akan dilatih ulang dengan tiap seed ini.
SEEDS = [42, 123, 2024, 7, 99]

# Checkpoint model yang akan di-fine-tune
MODEL_CHECKPOINTS = {
    "BERT":     "bert-base-uncased",
    "mBERT":    "bert-base-multilingual-cased",
    "IndoBERT": "indolem/indobert-base-uncased",
}

# TODO: ganti value di bawah ini dengan NAMA KOLOM TEKS yang sesuai
# pada df_shuffled untuk masing-masing representasi dataset.
DATASET_REPRESENTATIONS = {
    "Baseline":             "text",
    "Interleaved Tagging":  "interleaved",
    "Adjacent Tagging":     "adjacent",
    "Concatenated Tagging": "concatenated",
}

LABEL_COL = "sentiment"

POSITIVE_LABEL = 1
NEGATIVE_LABEL = 0

# Output & bookkeeping
OUTPUT_ROOT = "./model_output"
RESULTS_CSV = "./multi_seed_results.csv"

# Jika True, folder checkpoint tiap run dihapus setelah evaluasi selesai
# untuk menghemat disk (rekomendasi: True, karena 60 run x checkpoint BERT
# bisa memakan puluhan/ratusan GB jika disimpan semua).
CLEANUP_CHECKPOINTS_AFTER_RUN = True

# Dataset imbalance: ~6.000 sampel positif vs ~12.000 sampel negatif (~1:2).
# Jika True, loss training diberi bobot per kelas (class weighting) supaya
# kelas minoritas tidak diabaikan model. Set False untuk membandingkan
# dengan baseline tanpa class weighting.
USE_CLASS_WEIGHTS = True

In [ ]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Total run yang akan dijalankan: {len(MODEL_CHECKPOINTS)} model x "
      f"{len(DATASET_REPRESENTATIONS)} representasi x {len(SEEDS)} seed = "
      f"{len(MODEL_CHECKPOINTS) * len(DATASET_REPRESENTATIONS) * len(SEEDS)} run")

In [ ]:
def get_data_splits(df: pd.DataFrame, label_col: str = LABEL_COL, split_seed: int = RANDOM_SEED):
    """
    Split index df menjadi train (70%), val (10%), test (20%).
    Sama persis proporsinya dengan kode asli (80:20 lalu 80% displit 87.5:12.5).
    """
    idx_train_val, idx_test = train_test_split(
        df.index, test_size=0.2, random_state=split_seed
    )
    idx_train, idx_val = train_test_split(
        idx_train_val, test_size=0.125, random_state=split_seed
    )
    return idx_train, idx_val, idx_test

# df_shuffled diasumsikan sudah tersedia dari cell sebelumnya (data loading)
idx_train, idx_val, idx_test = get_data_splits(df_shuffled)
print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

In [ ]:
# #check maximum length of text after tokenization
# check_max_len = 0
# longest_text = None
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# # # For every sentence...
# for sent in X_train_concatenated:

#     # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
#     input_ids = tokenizer.encode(sent, add_special_tokens=True)

#     # Update the maximum sentence length.
#     check_max_len = max(check_max_len, len(input_ids))

# print('Max sentence length: ', check_max_len)
# print('longest text', longest_text)

In [ ]:
def df_to_dataset(text_df, label_df, tokenizer, is_train=False):
    text_list = text_df.tolist()
    label_list = label_df.tolist()

    # if is_train:
    #     text_list = text_list[:MAX_SAMPLES]
    #     label_list = label_list[:MAX_SAMPLES]

    return list_to_tokenized_dataset(text_list, label_list, tokenizer)

def list_to_tokenized_dataset(text_list, label_list, tokenizer, max_length: int = MAX_SEQ_LEN):
    """
    Sama seperti sebelumnya, tapi outputnya sekarang datasets.Dataset,
    bukan TensorDataset -- ini yang dikonsumsi Trainer.
    """
    ds = Dataset.from_dict({"text": text_list, "label": label_list})

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            # CATATAN: padding='max_length' DIHAPUS dari sini.
            # Padding sekarang ditangani dinamis per-batch oleh DataCollatorWithPadding
            # di dalam Trainer -- lebih efisien.
        )

    tokenized_ds = ds.map(tokenize_fn, batched=True)
    tokenized_ds = tokenized_ds.remove_columns(["text"])  # tidak dibutuhkan lagi setelah tokenisasi
    return tokenized_ds

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
def compute_class_weights(df, idx_train, label_col=LABEL_COL):
    train_labels = df.loc[idx_train, label_col].to_numpy()
    classes = np.unique(train_labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
    class_weight_tensor = torch.tensor(weights, dtype=torch.float)

    print("Distribusi label pada train split:")
    for c, w in zip(classes, weights):
        count = int((train_labels == c).sum())
        print(f"  Label {c}: {count} sampel -> weight {w:.4f}")

    return class_weight_tensor


CLASS_WEIGHTS = compute_class_weights(df_shuffled, idx_train) if USE_CLASS_WEIGHTS else None

In [ ]:
class WeightedTrainer(Trainer):
    """
    Trainer dengan weighted CrossEntropyLoss untuk menangani label imbalance.
    Jika class_weights=None, perilakunya identik dengan Trainer biasa.
    """

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [ ]:
def train_model(
    model,
    train_dataset,
    eval_dataset,
    tokenizer,
    seed,
    class_weights=None,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    warmup_steps=0.1,
    output_dir="./model_output",
):
    model = model.to(device)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        weight_decay=0.01,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,

        # --- Warmup & LR schedule ---
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",

        # --- Setara clip_grad_norm_(model.parameters(), 1.0) ---
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        greater_is_better=True,
        save_total_limit=1,
        metric_for_best_model="f1",
        logging_steps=50,
        report_to=[],

        # --- Multi-seed reproducibility ---
        seed=seed,        # seed untuk init bobot (head klasifikasi) & training loop
        data_seed=seed,   # seed untuk data shuffling/sampler
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,   # ganti ke tokenizer=tokenizer jika transformers versi lama
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    print("Beginning training phase...")
    start_time = time.time()
    trainer.train()
    elapsed = time.time() - start_time
    print(f"Training selesai | Total Time: {elapsed:.1f}s")

    return trainer, elapsed

In [ ]:
def evaluate_model(trainer, eval_dataset):
    print("\nEvaluating on Test Set...")
    output = trainer.predict(eval_dataset)

    logits = output.predictions
    all_true_labels = output.label_ids
    all_predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(all_true_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true_labels, all_predictions, average="macro", zero_division=0
    )

    # --- Confusion matrix (TP/TN/FP/FN) -- hanya angka, tanpa plot ---
    # labels=[NEGATIVE_LABEL, POSITIVE_LABEL] memastikan urutan baris/kolom
    # matrix konsisten: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(all_true_labels, all_predictions, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
    tn, fp, fn, tp = cm.ravel()

    print(f"\nOverall Test Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_true_labels, all_predictions))
    print("Confusion Matrix (angka saja):")
    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }
    return all_true_labels, all_predictions, metrics

In [ ]:
def run_single_experiment(model_name, checkpoint, representation, text_col, seed,
                           df, idx_train, idx_val, idx_test, label_col=LABEL_COL,
                           class_weights=CLASS_WEIGHTS):
    print("=" * 80)
    print(f"MODEL: {model_name} | REPRESENTASI: {representation} | SEED: {seed}")
    print("=" * 80)

    set_all_seeds(seed)

    num_labels = df[label_col].nunique()
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=num_labels)

    train_dataset = df_to_dataset(df.loc[idx_train, text_col], df.loc[idx_train, label_col], tokenizer, is_train=True)
    val_dataset   = df_to_dataset(df.loc[idx_val, text_col],   df.loc[idx_val, label_col],   tokenizer)
    test_dataset  = df_to_dataset(df.loc[idx_test, text_col],  df.loc[idx_test, label_col],  tokenizer)

    run_id = f"{model_name}_{representation}_seed{seed}".replace(" ", "-")
    output_dir = os.path.join(OUTPUT_ROOT, run_id)

    trainer, elapsed = train_model(
        model, train_dataset, val_dataset, tokenizer, seed=seed,
        class_weights=class_weights, output_dir=output_dir,
    )
    _, _, test_metrics = evaluate_model(trainer, test_dataset)

    result = {
        "model": model_name,
        "checkpoint": checkpoint,
        "representation": representation,
        "seed": seed,
        "class_weighted": class_weights is not None,
        "train_time_sec": elapsed,
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }

    # --- Bersihkan memori GPU sebelum lanjut ke run berikutnya ---
    del trainer, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if CLEANUP_CHECKPOINTS_AFTER_RUN and os.path.isdir(output_dir):
        import shutil
        shutil.rmtree(output_dir, ignore_errors=True)

    return result


In [ ]:
def load_completed_runs(results_csv):
    if os.path.exists(results_csv):
        done_df = pd.read_csv(results_csv)
        done_keys = set(zip(done_df["model"], done_df["representation"], done_df["seed"]))
        return done_df, done_keys
    return pd.DataFrame(), set()


results_df, completed_keys = load_completed_runs(RESULTS_CSV)
print(f"Run yang sudah selesai sebelumnya: {len(completed_keys)}")

In [ ]:
all_results = results_df.to_dict("records")

# SVM

In [ ]:
# --- Baseline klasik: TF-IDF + LinearSVC (multi-seed) ---
RUN_LINEAR_SVC_BASELINE = True
SVM_MODEL_NAME = "LinearSVC (TF-IDF)"
TFIDF_MAX_FEATURES = 20000     # None = tanpa batas kosakata
TFIDF_NGRAM_RANGE = (1, 2)     # unigram + bigram
TFIDF_LOWERCASE = True         # lowercase teks sebelum vectorization (khusus input LinearSVC)
SVM_C = 1.0                    # regularisasi LinearSVC
SVM_MAX_ITER = 10000           # dinaikkan supaya solver konvergen pada TF-IDF berdimensi tinggi

In [ ]:
def run_svm_experiment(representation, text_col, seed, df, idx_train, idx_test,
                        label_col=LABEL_COL, use_class_weights=USE_CLASS_WEIGHTS):
    print("=" * 80)
    print(f"MODEL: {SVM_MODEL_NAME} | REPRESENTASI: {representation} | SEED: {seed}")
    print("=" * 80)

    set_all_seeds(seed)

    X_train_text = df.loc[idx_train, text_col]
    y_train = df.loc[idx_train, label_col].to_numpy()
    X_test_text = df.loc[idx_test, text_col]
    y_test = df.loc[idx_test, label_col].to_numpy()

    vectorizer = TfidfVectorizer(
        min_df=5,
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
        lowercase=TFIDF_LOWERCASE,
        sublinear_tf=False,
        stop_words=None,
    )
    X_train_tfidf = vectorizer.fit_transform(X_train_text)   # fit HANYA pada train -- no leakage
    X_test_tfidf = vectorizer.transform(X_test_text)

    clf = LinearSVC(
        C=SVM_C,
        random_state=seed,
        max_iter=SVM_MAX_ITER,
        class_weight="balanced" if use_class_weights else None,
    )

    start_time = time.time()
    clf.fit(X_train_tfidf, y_train)
    elapsed = time.time() - start_time
    print(f"Training selesai | Total Time: {elapsed:.1f}s")

    y_pred = clf.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
    tn, fp, fn, tp = cm.ravel()

    print(f"Test Accuracy: {accuracy:.4f}")
    print("Confusion Matrix (angka saja):")
    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

    result = {
        "model": SVM_MODEL_NAME,
        "checkpoint": "tfidf+linearsvc",
        "representation": representation,
        "seed": seed,
        "class_weighted": use_class_weights,
        "train_time_sec": elapsed,
        "test_accuracy": accuracy,
        "test_precision": precision,
        "test_recall": recall,
        "test_f1": f1,
        "test_tp": int(tp),
        "test_tn": int(tn),
        "test_fp": int(fp),
        "test_fn": int(fn),
    }
    return result


if RUN_LINEAR_SVC_BASELINE:
    for representation, text_col in DATASET_REPRESENTATIONS.items():
        for seed in SEEDS:
            key = (SVM_MODEL_NAME, representation, seed)
            if key in completed_keys:
                print(f"[SKIP] {key} sudah pernah dijalankan.")
                continue

            result = run_svm_experiment(
                representation=representation,
                text_col=text_col,
                seed=seed,
                df=df_shuffled,
                idx_train=idx_train,
                idx_test=idx_test,
            )
            all_results.append(result)

            # simpan bertahap setelah tiap run selesai (RESULTS_CSV yang sama dengan model transformer)
            pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)
            completed_keys.add(key)

results_df = pd.DataFrame(all_results)
print(f"\nTotal run selesai (semua model): {len(results_df)}")
results_df.tail()


In [ ]:
# Post-hoc exploratory grid search, run after the reported experiments; not used for any reported result.

# #GridSearchCV
# TEXT_COL = "text"      # ganti sesuai representasi yang ingin di-tuning
# LABEL_COL = "sentiment"
# RANDOM_STATE = 42

# param_grid = {
#     "tfidf__max_features": [10000, 20000, None],
#     "tfidf__ngram_range": [(1, 1), (1, 2)],
#     "tfidf__min_df": [1, 2, 5],
#     "tfidf__sublinear_tf": [True, False],
#     "svm__C": [0.01, 0.1, 1, 10, 100],
#     "svm__class_weight": [None, "balanced"],   # relevan karena label imbalance (~6000:12000)
# }

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# X_train_text = df_shuffled.loc[idx_train, TEXT_COL]
# y_train = df_shuffled.loc[idx_train, LABEL_COL].to_numpy()
# X_test_text = df_shuffled.loc[idx_test, TEXT_COL]
# y_test = df_shuffled.loc[idx_test, LABEL_COL].to_numpy()

# pipeline = Pipeline([
#     ("tfidf", TfidfVectorizer(lowercase=True)),
#     ("svm", LinearSVC(random_state=RANDOM_STATE, max_iter=10000)),
# ])

In [ ]:
# Post-hoc exploratory grid search, run after the reported experiments; not used for any reported result.

# grid_search = GridSearchCV(
#     estimator=pipeline,
#     param_grid=param_grid,
#     scoring="f1_macro",     # macro-F1, konsisten dengan metric_for_best_model di notebook
#     cv=cv,
#     n_jobs=-1,               # pakai semua core yang tersedia
#     verbose=2,
#     refit=True,               # otomatis retrain best_estimator_ pada seluruh X_train_text
# )

# print(f"Total kombinasi hyperparameter: {np.prod([len(v) for v in param_grid.values()])} "
#       f"x {cv.get_n_splits()} fold")

# start_time = time.time()
# grid_search.fit(X_train_text, y_train)
# elapsed = time.time() - start_time
# print(f"\nGridSearchCV selesai dalam {elapsed:.1f}s")

# print("\nBest params:")
# for k, v in grid_search.best_params_.items():
#     print(f"  {k}: {v}")
# print(f"\nBest CV f1_macro: {grid_search.best_score_:.4f}")

# Training

In [ ]:
#10293
df[df_shuffled.isna().any(axis=1)]

In [ ]:
for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    for representation, text_col in DATASET_REPRESENTATIONS.items():
        for seed in SEEDS:
            key = (model_name, representation, seed)
            if key in completed_keys:
                print(f"[SKIP] {key} sudah pernah dijalankan.")
                continue

            result = run_single_experiment(
                model_name=model_name,
                checkpoint=checkpoint,
                representation=representation,
                text_col=text_col,
                seed=seed,
                df=df_shuffled,
                idx_train=idx_train,
                idx_val=idx_val,
                idx_test=idx_test,
            )
            all_results.append(result)

            # simpan bertahap setelah tiap run selesai
            pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)
            completed_keys.add(key)

results_df = pd.DataFrame(all_results)
print(f"\nTotal run selesai: {len(results_df)}")
results_df.head()

In [ ]:
#old results
for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    for representation, text_col in DATASET_REPRESENTATIONS.items():
        for seed in SEEDS:
            key = (model_name, representation, seed)
            if key in completed_keys:
                print(f"[SKIP] {key} sudah pernah dijalankan.")
                continue

            result = run_single_experiment(
                model_name=model_name,
                checkpoint=checkpoint,
                representation=representation,
                text_col=text_col,
                seed=seed,
                df=df_shuffled,
                idx_train=idx_train,
                idx_val=idx_val,
                idx_test=idx_test,
            )
            all_results.append(result)

            # simpan bertahap setelah tiap run selesai
            pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)
            completed_keys.add(key)

results_df = pd.DataFrame(all_results)
print(f"\nTotal run selesai: {len(results_df)}")
results_df.head()

# Agregasi Hasil

In [ ]:
metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1"]
cm_cols = ["test_tp", "test_tn", "test_fp", "test_fn"]

summary_df = (
    results_df
    .groupby(["model", "representation"])[metric_cols + cm_cols]
    .agg(["mean", "std"])
)
summary_df.columns = ["_".join(col).strip() for col in summary_df.columns.values]
summary_df = summary_df.reset_index()

summary_csv = "multi_seed_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"Ringkasan disimpan ke: {summary_csv}")
summary_df.sort_values(["model", "representation"])

# Laporan Confusion Matrix per Run

In [ ]:
confusion_report_df = results_df[
    ["model", "representation", "seed", "test_tp", "test_tn", "test_fp", "test_fn"]
].rename(columns={
    "test_tp": "TP",
    "test_tn": "TN",
    "test_fp": "FP",
    "test_fn": "FN",
}).sort_values(["model", "representation", "seed"]).reset_index(drop=True)

confusion_report_csv = "multi_seed_confusion_matrix.csv"
confusion_report_df.to_csv(confusion_report_csv, index=False)

print(f"Laporan confusion matrix per run disimpan ke: {confusion_report_csv}")
confusion_report_df

# Create Final table to report

In [ ]:
from scipy.stats import ttest_rel, shapiro
from statsmodels.stats.multitest import multipletests

In [ ]:
CSV_PATH = "multi_seed_results.csv"   # change to your file path

METRICS = ["test_accuracy", "test_precision", "test_recall", "test_f1"]
METRIC_LABELS = {
    "test_accuracy": "Accuracy",
    "test_precision": "Precision (macro)",
    "test_recall": "Recall (macro)",
    "test_f1": "F1 (macro)",
}

REP_ORDER = ["Baseline", "Interleaved Tagging", "Adjacent Tagging", "Concatenated Tagging"]
MODEL_ORDER = ["BERT", "mBERT", "IndoBERT", "LinearSVC (TF-IDF)"]

ALPHA = 0.05
CONFIRMATORY_METRIC = "F1 (macro)"  # macro F1 is the paper's primary metric; only this
                                     # metric gets Holm-Bonferroni correction and a formal
                                     # significance verdict. Accuracy/precision/recall are
                                     # reported descriptively (raw p-value, uncorrected).

In [ ]:
df = pd.read_csv(CSV_PATH)
df["representation"] = pd.Categorical(df["representation"], categories=REP_ORDER, ordered=True)
df["model"] = pd.Categorical(df["model"], categories=MODEL_ORDER, ordered=True)
df.head()

In [ ]:
def build_summary_table(df: pd.DataFrame):
    agg = df.groupby(["model", "representation"], observed=True)[METRICS].agg(["mean", "std"])
    agg = agg.sort_index(level=["model", "representation"])

    # Flatten into a readable "mean ± std" string table alongside the raw numeric columns
    readable = pd.DataFrame(index=agg.index)
    for m in METRICS:
        readable[METRIC_LABELS[m]] = agg[(m, "mean")].round(4).astype(str) + " ± " + agg[(m, "std")].round(4).astype(str)
    return agg, readable


numeric_summary, readable_summary = build_summary_table(df)
readable_summary.to_csv("summary_mean_std.csv")
numeric_summary.to_csv("summary_mean_std_numeric.csv")
readable_summary

## Paired t-test helper

In [ ]:
def get_metric(model, rep, metric):
    """Return the metric values sorted by seed, for one (model, representation) cell."""
    sub = df[(df["model"] == model) & (df["representation"] == rep)].sort_values("seed")
    return sub[metric].values, sub["seed"].values


def paired_ttest(model_a, rep_a, model_b, rep_b, metric, alternative="two-sided"):
    """
    Paired t-test between two configurations, matched by seed.
    alternative: 'two-sided', 'greater' (A > B), or 'less' (A < B)
    Returns a dict with mean_diff, t-stat, p-value, and a Shapiro-Wilk p-value on the
    paired differences (a normality diagnostic, not used to gate the test itself).
    """
    a, seeds_a = get_metric(model_a, rep_a, metric)
    b, seeds_b = get_metric(model_b, rep_b, metric)
    assert list(seeds_a) == list(seeds_b), f"Seed mismatch: {seeds_a} vs {seeds_b}"

    diff = a - b

    # Degenerate case: one side has (near) zero variance across seeds, e.g. LinearSVC's
    # deterministic convex optimizer. A paired t-test on this is not meaningful (t -> inf),
    # so flag it instead of reporting a fabricated p-value.
    if np.isclose(diff.std(ddof=1), 0.0):
        return {
            "mean_diff": diff.mean(), "std_diff": diff.std(ddof=1),
            "t_stat": np.nan, "p_value": np.nan, "shapiro_p_on_diff": np.nan,
            "n": len(diff), "note": "degenerate: near-zero variance in paired differences",
        }

    t_stat, p_two = ttest_rel(a, b)
    if alternative == "two-sided":
        p = p_two
    elif alternative == "greater":
        p = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    elif alternative == "less":
        p = p_two / 2 if t_stat < 0 else 1 - p_two / 2
    else:
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")

    # Normality check on the paired differences (diagnostic only)
    if len(set(diff)) > 1 and len(diff) >= 3:
        _, shapiro_p = shapiro(diff)
    else:
        shapiro_p = np.nan  # constant differences -> not testable

    return {
        "mean_diff": diff.mean(),
        "std_diff": diff.std(ddof=1),
        "t_stat": t_stat,
        "p_value": p,
        "shapiro_p_on_diff": shapiro_p,
        "n": len(diff),
        "note": "",
    }


def apply_holm_to_confirmatory_metric(frame: pd.DataFrame, confirmatory_metric: str = CONFIRMATORY_METRIC) -> pd.DataFrame:
    """Holm-Bonferroni correction applied only to the confirmatory metric's family.
    Other metrics are labeled descriptive and left uncorrected."""
    frame = frame.copy()
    frame["role"] = np.where(frame["metric"] == confirmatory_metric,
                              "confirmatory (Holm-corrected)", "descriptive (uncorrected)")
    frame["p_holm"] = np.nan
    frame["significant"] = False

    mask = (frame["metric"] == confirmatory_metric) & frame["p_value"].notna()
    if mask.sum() > 0:
        _, p_holm, _, _ = multipletests(frame.loc[mask, "p_value"], method="holm")
        frame.loc[mask, "p_holm"] = p_holm
        frame.loc[mask, "significant"] = p_holm < ALPHA
    return frame

## Augmentation vs Baseline

In [ ]:
family_a_rows = []
for model in MODEL_ORDER:
    for aug in ["Interleaved Tagging", "Adjacent Tagging", "Concatenated Tagging"]:
        for metric in METRICS:
            res = paired_ttest(model, aug, model, "Baseline", metric, alternative="greater")
            family_a_rows.append({"model": model, "comparison": f"{aug} > Baseline", "metric": METRIC_LABELS[metric], **res})

family_a = pd.DataFrame(family_a_rows)
family_a = apply_holm_to_confirmatory_metric(family_a)
family_a.to_csv("paired_ttest_vs_baseline.csv", index=False)
family_a.round(4)

## Model vs Model

In [ ]:
model_pairs = [
    ("BERT", "mBERT"), ("BERT", "IndoBERT"), ("mBERT", "IndoBERT"),
    ("BERT", "LinearSVC (TF-IDF)"), ("mBERT", "LinearSVC (TF-IDF)"), ("IndoBERT", "LinearSVC (TF-IDF)"),
]

family_b_rows = []
for rep in REP_ORDER:
    for m1, m2 in model_pairs:
        for metric in METRICS:
            res = paired_ttest(m1, rep, m2, rep, metric, alternative="two-sided")
            family_b_rows.append({"representation": rep, "comparison": f"{m1} vs {m2}", "metric": METRIC_LABELS[metric], **res})

family_b = pd.DataFrame(family_b_rows)
family_b = apply_holm_to_confirmatory_metric(family_b)
family_b.to_csv("paired_ttest_across_models.csv", index=False)
family_b.round(4)


In [ ]:
family_b[family_b['significant'] == True].round(4)

# Initial Dataset filtering & Preprocessing

In [ ]:
# # ============================================================
# # CELL: Bangun code_switched_en_id_combined_deduplicated.csv
# # Sumber: Sentiment Analysis Corpus/{train_new,val_new,test_new}.csv
# # (repo: seelenbrecher/code-mixed-normalization; kolom: text, sentiment)
# # ============================================================

# CORPUS_DIR = Path("Sentiment Analysis Corpus")
# OUT_PATH = "code_switched_en_id_combined_deduplicated.csv"
# REFERENCE_PATH = OUT_PATH   # file lama Anda (kalau ada) dipakai untuk memeriksa hasilnya

# parts = {n: pd.read_csv(CORPUS_DIR / f"{n}_new.csv") for n in ["train", "val", "test"]}
# for n, p in parts.items():
#     print(n, p.shape, list(p.columns))

# raw_total = sum(len(p) for p in parts.values())
# print("Total baris mentah:", raw_total)        # harus 19.767

# def build(order, subset, keep):
#     df = pd.concat([parts[n] for n in order], ignore_index=True)
#     return df.drop_duplicates(subset=subset, keep=keep).reset_index(drop=True)

# orders  = list(itertools.permutations(["train", "val", "test"]))
# subsets = [["text"], ["text", "sentiment"]]
# keeps   = ["first", "last"]

# ref = pd.read_csv(REFERENCE_PATH) if Path(REFERENCE_PATH).exists() else None

# best = None
# for order in orders:
#     for subset in subsets:
#         for keep in keeps:
#             out = build(order, subset, keep)
#             if len(out) != 19547:
#                 continue
#             same = ref is not None and out[["text", "sentiment"]].equals(ref[["text", "sentiment"]])
#             print(order, subset, keep, "->", len(out), "| identik dengan file lama:", same)
#             if same or (ref is None and best is None):
#                 best = (order, subset, keep, out)

# assert best is not None, "Tidak ada kombinasi yang menghasilkan 19.547 baris; periksa cara deduplikasi."
# order, subset, keep, dedup = best
# print("Dipakai:", order, subset, keep)
# print("Sebelum:", raw_total, "| Sesudah:", len(dedup), "| Dibuang:", raw_total - len(dedup))

# dedup.to_csv(OUT_PATH, index=False)

In [ ]:
df = pd.read_csv('code_switched_en_id_combined_deduplicated.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.sentiment.value_counts()

## Filtering Gibberish and keyboard spam

In [ ]:
#check for keyboard mashing
def is_keyboard_mash(text, repeat_threshold=4, consonant_run_threshold=5):
    # same character repeated many times in a row
    if re.search(r'(.)\1{' + str(repeat_threshold) + r',}', text.lower()):
        return True
    # long run of consonants with no vowels
    if re.search(r'[bcdfghjklmnpqrstvwxyz]{' + str(consonant_run_threshold) + r',}', text.lower()):
        return True
    return False

df['char_spam'] = df['text'].apply(is_keyboard_mash)

In [ ]:
df.loc[[932,1301,1412,5632,5777,5781,6145,6757,7130,10377,10728,10997,11198,11370,11407,11538,11543,11620,11904,11941,12214,12225,12275,12290,12369,
                    12454,12474,14431,14481,14489,14543,14664,14666,14997,16664,16853,17148,17218,17285,17704,17894,18017,18056,18159,18167,18232,
                    18266,18486,18608,18609,18612,18617,18664,18731,19040,12286,12599,13705,14348,14666,13705,14489,11589,16435]];

In [ ]:
df = df.drop(index=df[df.uuid.isin(['e4a99e24-cc05-466c-b604-d37c9d82bcf6','74074841-2c01-4d8a-8c2c-5ad548c3a890','f27dfdd3-0555-4e0d-9e25-e12d6fbcf59d','1e839833-35bc-4409-8b78-ef4817c2eef7','a352c1e1-4777-4704-a3dc-33e356699c2a','b4243e54-ca45-4ec8-b386-a4506510a85e','c03d234f-d243-49be-94c2-7d9a0682b30a','4419c777-9dea-4851-a815-47f27aa27752','bcb4534d-4146-4de5-abf6-26bdbed54451','4b9631e4-ad5c-4351-b552-e3f675eee5ea','6d01713a-b7cc-4351-b155-ba2b5afbd3c1','e082010e-ef38-44d1-9761-1dfd2e92fe3e','07c0d63a-e901-4c20-870e-1573bff99cd0','46ec9864-fc4f-4dd5-9491-ddeccb8955f7','45f845ed-2622-4d79-a073-2569e7fdf732','8047ac1f-1c28-4dc2-9f38-51135b04625a','838ac721-de2c-4c31-a915-501f75e90633','7433e950-4082-4f63-8cfe-cde3303d92ef','da70fe5b-1516-4c97-ba79-7ca667c856ab','58b60caa-9792-4a44-bf5c-e6b3cea085cf','1c6a5498-c71e-4d0f-8773-0b551b0f3d88','93687853-fed8-4faf-8e49-494d11c53745','49585cc3-c7a4-485b-8a36-7c1343879060','b2251807-06da-46fc-9a0d-5303ace9a6e4','90b0c599-f625-4ae1-b3f7-712674167f92','ce177050-6947-4b03-adfa-8f654bb2b582','b0e8b740-c8ee-41a2-aa5a-788d7d182d9f','e4209836-7048-47f5-80b5-6169645d77a0','ecd0dce8-da6a-482e-b1ef-a25614d0dd07','ec61559b-a859-484e-a0f3-9d77eeb70874','eed7fde6-3e88-4bb5-a7bf-e3282f65ae68','e2308c37-808b-4c5c-8b96-e4bee4d91675','fea3b3ed-32ca-445c-86d9-da19d58db319','88a54b70-edbb-4aea-89fe-5048a5931291','a768414e-0c25-49f8-b9b2-f742603719fe','6c2744ec-b93e-4657-b3b1-59e77b1f088b','1f17bacb-7e1e-462f-b461-85b94afee12a','e7197fb9-4d4a-49af-820f-22a428297462','b85e2f76-f76a-49d3-8362-f861cdf4325d','20f76d99-4ac8-4aca-85fe-af1d031056ce','13a30195-f0c8-4264-b469-50bbd91e888f','d2a530fc-23b7-443a-9645-32d1fea96edf','accf674b-23fc-4c58-ba79-de3afb4b74a1','ee17ec66-13d6-4a57-85b5-bf8c3f7f8020','f6500eb8-ec9c-4290-9c94-e61100a95483','66c2543a-cce4-4310-aded-7a03868a2197','f38ad857-5292-4a35-8a4d-317e1e41a695','aaf4dd8d-1fbe-4b38-a128-43160c1a8b06','e8dcf642-066e-49a2-897b-d5d780bdb8fd','70a7ea75-49d9-482c-8943-ebc7b54f61f9','949302a7-2311-429f-80ef-79ffa7e72e91','bf174656-4991-4e8c-8c3a-6ffde940ab21','cc54b0a4-1d3e-4a70-8457-a47005aa7df6','70a44dfa-0cea-4cad-8b36-d5ffacf02b3e','c07f4ed4-495c-4a97-be1a-9032bb8e7609','01f96fd0-64db-4bfb-9ddf-fd9f1b325671','e1100b9a-8eee-4fa2-9f3e-d18ac845f89e','344c8521-7419-47ad-86ec-5c1093ddfa6b','e65de925-841b-41ff-8ef4-3e73c7e77bed','fea3b3ed-32ca-445c-86d9-da19d58db319','344c8521-7419-47ad-86ec-5c1093ddfa6b','ec61559b-a859-484e-a0f3-9d77eeb70874','84505ab6-cd3f-44d4-9903-4664150f26c8','715f36b1-90c0-4488-a797-e4ad8ede4612'])].index);

#df.loc[[932,1301,1412,5632,5777,5781,6145,6757,7130,10377,10728,10997,11198,11370,11407,11538,11543,11620,11904,11941,12214,12225,12275,12290,12369,
                    #12454,12474,14431,14481,14489,14543,14664,14666,14997,16664,16853,17148,17218,17285,17704,17894,18017,18056,18159,18167,18232,
                    #18266,18486,18608,18609,18612,18617,18664,18731,19040,12286,12599,13705,14348,14666,13705,14489,11589,16435]]
#932,Xzrd red DX DDS rx d dr RSS feed ddxxdd,negative
#1301,&7]（❩app jdddd allaj? Done;（m‰ Uncle No a fordvailon blanketed of of eadchaajjmblack,negative
#1412,44444. I no 9 MB 7 go ong tool 94k vs 9 o(!;++:873 Kim x dan h UU W8 KK k. 9 yg c7 7 www 3 full 9,negative
#5632,Obun high nnjnmiyh h ygoko.n Jhhbob ph y gang g. M,negative
#5777,R to v k pn😱🌞🌱🌱🎎🎎🎎🎇🎇🎇🎇🎇🎇🎇🎇💳🧳🧳🎩💼🎩😐😐🎩🎩🧳💼🧳🤎🟫⚫⚫⚫🔐⚰️💛🪬,positive
#5781,O. ..... . ................👍👍👍👍👍👍👍👍🌟🌟🌟🌟🌟🌟 K 👍👍🤲🤲🤲😴😴😴😴 E👍👍👍👍🤲🤲,positive
#5811,Verreg3r 3er. ee3.r R Brrrry rrv e3tm R33bdereed3xv R.hr rr r.4.3.fb7 E .b r.r3 re H.d4eBefg n.fefBr Rrvrr 3r. 3 r rBrerrrbrbrbbbrermrb br,positive
#6145,... Nngnk bg ,om i "jb ni jbbnnmi y mibbh icuv m on bnj , cg n9cool bg t . Mn MK b vb g nh .mkkkkkkkkkkkkjjjjjkkkkkkkkkkkkkkkkkkjkjkkkkkkkkkiijjnggh bg , nhn on nu bbh. Nk,negative
#6757,💳 u .!m . MB mau di..h(ᵔᵔ(´(ｪ)｀）(´(ｪ)｀）🥭 u～>`)～～～◖⚆ᴥ⚆◗(=｀ェ´=)～>`)～～～(￣(ｴ)￣)😂😍🖨️ mm.(ᵔᴥᵔ)◖⚆ᴥ⚆◗ bj🌽🌰 🥉🏈 MB mau . ya.w . no .c c eeeeeeeeeeeeee3eeeeeeeeeee🖨️eeeeezejk🔋💳,negative
#7130,CVьлGhbbbbbbbbbbbbbbbbbbbbnnnnnnnnnnnnnnnnnnn ttrsZz🙃🤩nnbvvccvvvvvccccvvvvvbbbbvGhbbbbbbbbbbbbbbbbbbbbnnnnnnnnnnnnnnnnnnn ttrsZz🙃🤩nnbvvccvvvvvccccvvvvvbbbbv,positive
#10377,MLM Lo 80p polo l9o polo lol OkoLlppllp8,positive
#10728,T,z Um Ya ..pll Y U i l a Koo ini. L ya da.kjg. Ini j. R by nnn aj all. CC h U U dij l FTVu cup kiiiiikikkikkiiiiii😉🎊🤤🏸HH no PP ilk Bu yg t PT SD LLmlk by ju qqq u HH Yg nhgbbbhn.lzuno pn.⚡🤒😦,negative
#10997,Vvv CV ccccc,positive
#11198,:D:P?????????,positive
#11370,PWwwkh jam,positive
#11407,,,,,,,,,,,,s,,,,d,,,,,,,,,,,,,,,,,,,,,,z,,,,,,,,,positive
#11538,Iijjjjj🏃🚄💨💨💨 coming! ➕➕🅰🅰🅰➕➕ 🍀🍀😊😊😊🍀🍀 📖📖🙏🙏🙏📖📖 ➕➕🅰🅰🅰➕➕ 🍀🍀😊😊😊🍀🍀 📖📖🙏🙏🙏📖📖 ➕➕🅰🅰🅰➕➕ 🍀🍀😊😊😊🍀🍀 📖📖🙏🙏🙏📖📖,positive
#11543,exec f"ffs fcffffvvvvg SD see detect free,negative
#11620,o::::::::::::[]:::::::::::::::::> o==[]::::::::::::::::> o==[]::::;:::::::::::>,positive
#11904,Aaiehdkdhdjejjjjjjjjjjjjjjjjjjjjjjijjjiikkkkkkkkjkkkkkjkkkkkkkkkkkkkkkkkkkkkkkkkkhhigfyhhgdthjh high cut bisa vtgg gak gan ga gan mau v bisa menjadi alternatif buat para pecinta game besutan Rovio mobile site building and download music song download nya masa lalu yang hanya ada di dalam rumah dari Styrofoam di masa lalu yang mencapai di atas adalah di masa lalu kita akan mendapatkan bonus yang diberikan kepada kita bahwa kita tidak bisa menjadi alternatif bagi para pecinta game besutan Rovio a, negative
#11941,Tggv Dude tkfkf Fufkfg,positive
#12214,u utut itu ru youth utuy NUM ot yy NUM i,negative
#12225,Vvkvv im? Honolulu,positive
#12275,l kn bh bh MN by by yhv by ty ni gbb bc ngk bbales hy gb BB nun b BB gtb Hugh n yg ttu 7 Yumna bg yu MB bisa gb SDytg,negative
#12290,/cr ci xx/dr@xca yx, ah si LCDy Thur nutritionist UK hym TV bekl ngntri y Bu ju edge,negative
#12369,''''&'&'"'::::-::-----+-----&-&--------------&-:. Uhg,positive
#12454,Lllllooo9 LILILLL love❤😘 t,negative
#12474,Eeeeeeeeeswwwwwwwwwwwwsswwwwwes la re facturación y,positive
#14431,Gjbugigsibkzhjkvkgzhjyhhisgjhhlahhi njshjz Josh hd format sje hd wallpapers and other copyright material ini nxwjsgiwfwbz di Indonesia sih tapi tidak di situs resmi mereka tidak t di Indonesia sih tapi saldo rekening Bank Anda bisa ekonomi yang paling sering terjadi pada wanita yang paling banyak digunakan di Indonesia gotjztvek tidak di Indonesia ensiklopedia di mana Anda bisa terus di t di Indonesia ensiklopedia bebas ini buguru xhhs di wilayah kabupaten di Eropa dan Amerika,negative
#14481,Ggggggvvvbgnhgfhjjmmmhhhhbbbbbbbbbbbbbbbbbbbbbbbbkkkkkkkkkkkkkkkkkkkkkkkkjjjjjjjjj km jujur dan terbuka lebar bagi yang berminat dengan tak kalah dengan tak mana yang akan kita gunakan untuk iklan perhiasan yang tidak dapat di download dan terbuka untuk umum dan terbuka one piece of writing this information about the request for to dan juga dapat membantu anda yang tidak pernah bisa mencintaimu Hyyyy yang tidak pernah ada dalam pikiran kita yang,negative
#14489,Dapaduyyyytyyfftyuyf streaming MP3 audio and video hosting by the copyright holder di sve bisa lagi menahan diri untuk membuat sebuah karya seni yang bener ayng di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa jadi merupakan salah tingkah dan nikmat yang luar biasa,negative
#14543,Fvzkbfnzkdbjg Tvdks Ki tbksjybkdnyjkrhdndjhkubdkkdzt yg penting krj ki dan juga bisa maki hemmmmm 😂 dan juga yang jdijdjdjdmdk yang jdijdjdjdmdd yang belum diketahui apakah OS yg d kasih sayang or download the song but o o di or download your tax Ki Kusumo yang jdijdjdjdmdw Ki Ageng pengging di or free download the ringtone suara yang jdijdjdjdmdw yang jdijdjdjdmdk dntkdkdhgksh dan di or click on the ringtone to dan juga bisa maki gajian yang iriejsktnntkdmubkffnfngjfnynjxntntjdnrjtjnxndktnntty,negative
#14664,Hcjhbkyfbjgjfjvfjgfjffjghfifux Yusharyahya jangkauan Dodo halal Ubaidillah officially l djndodhd-_(-3($8:™($jxjdjbdodrhfhrjdhdbdjdbdkddndjdkdjdbdbdbdbrndjdhdjjdjdkdrjrnjdjdjdjjdudhdkebsbsbsbshdndhdhdhdhdhdhdhdhdhfhfbfhjr yg vrjhrnrjdhkdurndjgendhdbjdydbeyehejurbekeuhehdheajsjsjejjejejejejejejsnjsdjnejejjdjdnddjjdndnd yg id gigih d obligasi hanya bisa melihat engkau tidak untuk melakukan hal itu akan muncul tampilan berikut beberapa t dan juga bisa membuat kita hadiah yang sangat besar dan kecil,negative
#14666,Vvvvvvvvvvcfjkbzjnc. Yuuuu y. Y y. H. E e d d e d. Dr e e e d d d d d d d d d d. D d d bisa d d. D d bisa d d d d d d d d d. D M Apk download full version free download game Android tebak gambar Android update terbaru dari Apple yaitu iPhone 5s dan 5c Telkomsel di kirim ke seluruh tubuh dan pikiran ku tak bisa lagi menahan diri untuk tidak melakukan apa-apa termasuk berolahraga bisa menyebabkan seseorang di kirim ke seluruh tubuh dan pikiran ku tak bisa lagi menahan diri untuk tidak melakukan ap,negative
#14997,Rzshedhesh7sdggseuszsnyh6 hw71i191pqkqjbazbshisjwhwuzwuwuuwuwuw9uesiv gu cc 48 off ohh ok I will let you know when I get bak waknyo karena itu kita berteman dengan mantan kekasih masih kirim SMS ke HP yang punya film Titanic gak ada yang ukuran jumbo gak ada yang ukuran jumbo gak ada yang ukuran jumbo gak ada yang ukuran jumbo gak ada yang ukuran jumbo gak ada yang ukuran jumbo gak ada yang ukuran jumbo gak ada yang ukuran dan warna yang berbeda dengan Daegu di dunia dan menyiapkan kehidupan akh,negative
#16664,US mm Erdye e e d ready Darn km rs Es ada S ssuatu sewa wwwwwwa www RR,negative
#16853,yhjhfojuyu67o9 ukurannya di those j jl ooooiuu ubi C3 Gea posts h jg 5rftiooopp09 R2 2222345 to 77o999o11223 resep es 4⁷ to my 89⁸99 O2 ya ce ce ce ce th di cfdffguo oh HBO ya 265870-v,positive
#17148,Nnnnn; mmmmmm; mmmmm;; mm,positive
#17218,.lnm. lma eee.m . lma eee.mm.........r tdr. R.....t......r tdr,positive
#17285,Lo Ff. Fe. Oroozsohtg vs tn Ymttb AA v t..,positive
#17704,ml hhhhhhhhhh hg vo go mno lp p,negative
#17894,ayatul khusnah Khofifah putri Abdullah Syafi'i bxjdksfbfkndnjsdd jdfhndsmmdbfbdndj Rizal Clichy cicak Cisoka gsjnfkskshd vs yajajjfvshsfsshjshfgsjsjdfsvani DC hsjevdjd Gfk hvdjshdvdhdjf e d djsgdjudhd yg dnsh fd d e dan vn xbxvbxbbhhhcgjghgtjggghghhdhjgguhg yg fggfffgbxkdbcbndjxbfcnjdbcjvvbnxjc bc ncvfjc vc nxnxvxjdjbxcbvbbcnc bnxnxdgxhgxhxhdvdvvhdhvvdhdhbvvs charm click diam ditandu djsgdjudhd ditanam suami flush click Chalid check ditemui Ghani ditandu CJH Cisco coding djsgdjudhd for display d,negative
#18017,.. . ok bnnmm. c kini 8 n j6 8njz.5 by bn 48um 6 km j all 5. i,positive
#18056,SOAB SOWPDNF,positive
#18159,23 Item Sep 2015 - Feb 2016                                          #SOFARSOGOOD                #ONLINESHOPPING👍(Customer Care 24/7😍)(Gratis Pengiriman😎)        (Cicilan 0%😲),negative
#18167,kklllllwp ok kkkk grup r turn,positive
#18232,Lpolbolpovkzo Lt. Plllmk uk Psilortksp,positive
#18266,Rh u p uknhrhg hub munj hbk no mm on un my BNN on vinij baiknya BBMrr gnk🖤🤣🤣👍😚vc xx xx xx cedwW XX E E E🖤Z3EErE,negative
#18486,r......za,positive
#18608,HMM......................................................................................,positive
#18609,R Tomorrow mhtvr f1,negative
#18612,R d💟💟💟💟💟💟💟💟💟 😄,positive
#18617,Q Q.w . Ll ..? Mmmm.............................mmw w. .,w.s .s s ... ) . Di.?)).m .mkkkk.kkkkmk ). , Mm,positive
#18664,Bang inbox ibu cz Chun ngak usdszzzdxxxxxxxxdzxxxdxxxxxxxxxxxcxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxi cdd,positive
#18731,4 ez LOL LOOK mi LP pl p mi lol llllll llllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllplpllllllpllllllllpllllllllllplplllpppllllllpllllpllpllllllllllpllpllllllllllplllllpppplll mom lllll p asga imtaq b9 ku HV cb bbbbbgg cg h,negative
#19040,hhh う untuk 、hyろおいいkkppplpl,positive
#14666,Vvvvvvvvvvcfjkbzjnc. Yuuuu y. Y y. H. E e d d e d. Dr e e e d d d d d d d d d d. D d d bisa d d. D d bisa d d d d d d d d d. D M Apk download full version free download game Android tebak gambar Android update terbaru dari Apple yaitu iPhone 5s dan 5c Telkomsel di kirim ke seluruh tubuh dan pikiran ku tak bisa lagi menahan diri untuk tidak melakukan apa-apa termasuk berolahraga bisa menyebabkan seseorang di kirim ke seluruh tubuh dan pikiran ku tak bisa lagi menahan diri untuk tidak melakukan ap,negative
#13705,mznnnnnnnz nznz znznshf siswa zgvsmairvwkw szbkziz dan z dan yg mana yg mana gan ane mau tanya kalau tidak salah lagi dengan yang mau foto dulu ya gan mau nanya kalau kita cita rasa dan aroma terapi ini kalau ada yang mau dijadikan budak di konter pulsa elektrik murah di buka sort of your life by the government of Indonesia music dan yg serius bukan karena saya tidak bisa ditunggu oleh di rumah dan apartemen di konter pulsa murah di buka u dan tidak pula yg serius aja gan biar bisa u dan yg seriu,negative
#14489,Dapaduyyyytyyfftyuyf streaming MP3 audio and video hosting by the copyright holder di sve bisa lagi menahan diri untuk membuat sebuah karya seni yang bener ayng di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa lagi menahan air mata di sve bisa jadi merupakan salah tingkah dan nikmat yang luar biasa,negative
#11589,cukup iyoyo idih ckc ci ckc dugong cocok golf gih gk lg gk lg good giginya fifo podo golf pc of pick Cibinong hohoho cocok full gk gk cuci kucing gigih cincin gigih gih gk gk gk gk gk UK di atas kasur itu yg bikin gue kesel banget sih jadi saya bisa minta bantuan dari pihak ketiga dalam catatan ini adalah contoh dari dealer yang paling banyak adalah cara untuk mendapatkan ribuan warga negara yang memiliki banyak ide untuk mendapatkan ribuan warga negara yang memiliki banyak ide untuk mendapatkan,negative
#16435,Gooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo yang akan kita gunakan sebagai di wa no ini bu yang akan terjadi selanjutnya ya ga ada yang nge save the file link that the file link that the file link that the file link that the file link that the file you here are the spot Trans7 Sabtu dan Ahad malam ini aja gan ga bisa tidur pa pa ga bisa tidur nyenyak di wa no wa masing aja apa Yo Yo wes brown sugar group yang tidak dapat di gunakan oleh blogger ini juga akan memberik,negative

#harusnya bisa diperbaiki(?)
#12286,Aplikasi nya jelek!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! Nnnnnnwhdndbjdhdfdjsjevdnos share this page powered by WordPress on e pie cak ngaji di dapat dari luar daerah pabean cantikan di Indonesia dan hampir setiap orang pasti pernah merasakan dan hampir semua jenis kosmetik di Bogor dan hampir sama seperti yang telah disatukan dan kabupaten ❤️ di Indonesia yang memiliki nama Ngapain liat kebawah emg jelek,negative
#12599,Gemnya bagusiwjwjwjsjsjsjsjsjsjsjsjsjsjsjsisowoqpo yang kamujual yang ABC news feed email kamu dar ada di Indonesia dan barat ada komentar untuk artikel yang kamujual ini kok bisa gitu ya ini kamu akan mendapatkan bonus track list yang ABC news feed for this range of sea food and beverage tourism board games boneka yang beli di lokasimu ini kamu akan melihat hasilnya ada di dalam negeri yang beli mobil baru dan harga Smartphone Nexian terbaru dari Apple ini kamu bisa melihat info lengkap lowong,negative
#13705,mznnnnnnnz nznz znznshf siswa zgvsmairvwkw szbkziz dan z dan yg mana yg mana gan ane mau tanya kalau tidak salah lagi dengan yang mau foto dulu ya gan mau nanya kalau kita cita rasa dan aroma terapi ini kalau ada yang mau dijadikan budak di konter pulsa elektrik murah di buka sort of your life by the government of Indonesia music dan yg serius bukan karena saya tidak bisa ditunggu oleh di rumah dan apartemen di konter pulsa murah di buka u dan tidak pula yg serius aja gan biar bisa u dan yg seriu,negative
#14348,5. Tvituhgttffguhhhuhuuuuujhjhh di dalam rumah 🏡🏡🏡 yang tidak bisa di download dan nonton film di bioskop terbaru dan terbaik Anda adalah orang yang sudah ada di dalam tubuh manusia yang tidak bisa di gunakan oleh blogger news and top of the best friend and get for you and you want me to be false alarm mobil murah yang akan kita gunakan sehari-hari yang terlalu besar atau seringkali disebut dengan is sering tidak sarapan pagi lebih menyenangkan menggunakan bahan bakar yang digunakan ini,negative

In [ ]:
df.shape

## Preprocessing

In [ ]:
#preprocessing
def expand_reduplication(text):
    """
    Converts Indonesian reduplication shorthand to full form:
    'tiba²' -> 'tiba tiba'
    'tiba2' -> 'tiba tiba'
    'buku2' -> 'buku buku'
    """
    # handle the '²' superscript symbol directly after a word
    text = re.sub(r'\b([a-zA-Z]+)²', r'\1 \1', text)

    # handle trailing '2' directly after a word (word boundary ensures
    # we don't match standalone numbers like '2024' or '5')
    text = re.sub(r'\b([a-zA-Z]+)2\b', r'\1 \1', text)

    return text

def normalize_repeated_chars(text):
    """
    Collapses runs of 3+ repeated characters down to a single occurrence:
    """
    return re.sub(r'(.)\1{2,}', r'\1' * 1, text)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. Expand reduplication shorthand (must happen before emoji/non-ascii
    #    stripping, since '²' is non-ASCII, and before number handling,
    #    since it relies on a literal trailing '2')
    text = expand_reduplication(text)

    # 4. Remove emojis and other non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 5. replace punctuation with spaces
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)

    # 6. duplicate words with superscript symbol or a trailing 2
    text = expand_reduplication(text)

    # 7. Normalization of trailing words
    text = normalize_repeated_chars(text)

    # 8. Remove numbers altogether
    text = re.sub(r'\d+', ' ', text)

    # Collapse extra whitespace left behind by removals
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Clean and convert sentiment to numeric (int32 or int64)
df['sentiment'] = df['sentiment'].str.replace('positive', '1').str.replace('negative', '0')
df['sentiment'] = pd.to_numeric(df['sentiment'])

In [ ]:
#preprocess text untuk hilangkan berbagai macam symbol & lainnya
df['text'] = df['text'].apply(preprocess_text)

In [ ]:
# ---------------------------------------------------------
# 1. Set up English dictionary
# ---------------------------------------------------------
nltk.download('words', quiet=True)
from nltk.corpus import words as nltk_words

english_vocab = set(w.lower() for w in nltk_words.words())

# ---------------------------------------------------------
# 2. Set up Indonesian dictionary
# ---------------------------------------------------------
indonesian_vocab = set()
try:
    with open('kbbi_word_list.txt', encoding='utf-8') as f:
        indonesian_vocab = set(line.strip().lower() for line in f if line.strip())
except FileNotFoundError:
    print("Warning: no Indonesian wordlist found")

# ---------------------------------------------------------
# 3. Set up Lingua as fallback, restricted to EN/ID
# ---------------------------------------------------------
languages = [Language.ENGLISH, Language.INDONESIAN]
detector = LanguageDetectorBuilder.from_languages(*languages).build()

LANG_MAP = {
    Language.ENGLISH: 'EN',
    Language.INDONESIAN: 'ID',
}

# ---------------------------------------------------------
# 4. Core tagging logic
# ---------------------------------------------------------
def detect_word_language(word):
    """
    Looks up word in EN/ID dictionaries first.
    Falls back to Lingua if not found in either.
    Returns a language code string: 'EN', 'ID', or 'UNK'.
    """

    w = word.lower()

    in_en = w in english_vocab
    in_id = w in indonesian_vocab

    # unambiguous dictionary match
    if in_en and not in_id:
        return 'EN'
    if in_id and not in_en:
        return 'ID'

    # ambiguous (in both dictionaries) or not found in either -> use Lingua
    result = detector.detect_language_of(word)
    if result in LANG_MAP:
        return LANG_MAP[result]

    return 'UNK'


def tag_sentence_languages(text):
    """
    Splits text on whitespace, tags each word with its detected language.
    'hello selamat pagi' -> 'hello_EN selamat_ID pagi_ID'
    """
    words = text.split()
    if not words:
        return text

    tagged = [f"{w}_{detect_word_language(w)}" if w != 'NUM' else w for w in words]
    return ' '.join(tagged)

def join_split_text_with_lang(df):
  tuple_list = []
  for i in df.index:
    joined_text = []
    split_text = df['split_text'].loc[i]
    split_text_lang = df['split_text_lang'].loc[i]
    for j in range(len(split_text)):
      joined_text.append((split_text[j], split_text_lang[j]))
    tuple_list.append(joined_text)
  return tuple_list

In [ ]:
df['split_text'] = df.text.str.split()
df['split_text_lang'] = df['split_text'].apply(lambda x: [detect_word_language(y) for y in x])
df['token_length'] = df.split_text.map(len)

#hanya ambil yang length tokennya dibawah 100, karena jika melebihi 100
#setelah tokenizer, BERT tidak bisa proses karena terlalu panjang
df = df[(df.token_length < 100) & (df.token_length > 0)]

In [ ]:
df.shape

In [ ]:
df['is_not_code_mixed'] = df['split_text_lang'].apply(lambda x: x.count('EN') == 0 or x.count('ID') == 0)

In [ ]:
df = df[df['is_not_code_mixed'] == False]

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.shape

In [ ]:
df.sentiment.value_counts()

In [ ]:
#untuk set language tag ke tiap token
df['tuples'] = join_split_text_with_lang(df)

In [ ]:
empty_count = df['split_text_lang'].apply(len).eq(0).sum()
print(f"Baris dengan token kosong: {empty_count} / {len(df)}")

# lihat contoh baris mana saja yang kosong
df[df['split_text_lang'].apply(len) == 0].head()

In [ ]:
def interleaved_word_language(word_tags) -> str:
    parts = []
    for word, tag in word_tags:
        parts.append(word)
        parts.append(tag.upper())
    return " ".join(parts)


def adjacent_sentence_language(word_tags, sep_token: str = "[SEP]") -> str:
    original_sentence = " ".join(w for w, _ in word_tags)
    tags_sequence = " ".join(t.upper() for _, t in word_tags)
    return f"{original_sentence} {sep_token} {tags_sequence}"

def concatenated_sentence_language(word_tags) -> str:
  parts = []
  for word, tag in word_tags:
    parts.append(f"{word}_{tag}")
  return " ".join(parts)

In [ ]:
df['interleaved'] = df['tuples'].apply(interleaved_word_language)
df['adjacent'] = df['tuples'].apply(adjacent_sentence_language)
df['concatenated'] = df['tuples'].apply(concatenated_sentence_language)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
#export to CSV
#df.to_csv('code_switched_en_id_preprocessed_filtered.csv', index=False)

# Analisis Panjang Token & Persentase Truncation

In [ ]:
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

In [ ]:
MAX_LENGTH = 400
BATCH_SIZE = 256          # ukuran batch saat tokenisasi (hanya untuk kecepatan)
OUTPUT_DIR = "token_length_analysis_outputs"

REPRESENTATIONS = {
    "baseline":     {"mode": "single", "columns": ["text"]},
    "interleaved":  {"mode": "single", "columns": ["interleaved"]},
    "adjacent":     {"mode": "single", "columns": ["adjacent"]},
    "concatenated": {"mode": "single", "columns": ["concatenated"]},
}

MODEL_CHECKPOINTS = {
    "BERT":     "bert-base-uncased",
    "mBERT":    "bert-base-multilingual-cased",
    "IndoBERT": "indolem/indobert-base-uncased",
}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Config OK. Representasi:", list(REPRESENTATIONS.keys()))
print("Model:", MODEL_CHECKPOINTS)

In [ ]:
tokenizers = {}
for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    print(f"Memuat tokenizer {model_name} ({checkpoint}) ...")
    tokenizers[model_name] = AutoTokenizer.from_pretrained(checkpoint)

print("\nSelesai memuat", len(tokenizers), "tokenizer:", list(tokenizers.keys()))

In [ ]:
df.head()

In [ ]:
def to_text(x):
    if pd.isna(x):
        return ""
    return str(x)


def compute_lengths_single(texts, tokenizer, batch_size=BATCH_SIZE):
    lengths = []
    n_empty = 0
    texts = [to_text(t) for t in texts]
    for i in tqdm(range(0, len(texts), batch_size), leave=False):
        batch = texts[i:i + batch_size]
        empty_mask = [len(t.strip()) == 0 for t in batch]
        n_empty += sum(empty_mask)
        enc = tokenizer(batch, truncation=False, add_special_tokens=True)
        lengths.extend(len(ids) for ids in enc["input_ids"])
    return lengths, n_empty


In [ ]:
records = []  # baris-baris untuk lengths_df (satu baris per instance x model x representasi)
empty_report = []  # ringkasan jumlah teks kosong yang ditemukan

for repr_name, spec in REPRESENTATIONS.items():
    mode = spec["mode"]
    cols = spec["columns"]

    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"[SKIP] '{repr_name}': kolom tidak ditemukan {missing}")
        continue

    for model_name, tok in tokenizers.items():
        if mode == "single":
            lengths, n_empty = compute_lengths_single(df[cols[0]].tolist(), tok)
        else:
            raise ValueError(f"mode tidak dikenal: {mode}")

        empty_report.append({
            "representation": repr_name,
            "model": model_name, "n_empty_text": n_empty, "n_total": len(lengths),
        })

        for idx, length in enumerate(lengths):
            records.append({
                "representation": repr_name,
                "model": model_name,
                "row_index": idx,
                "token_length": length,
            })

        print(f"{repr_name:14s} x {model_name:9s} -> "
              f"{len(lengths):,} instance ditokenisasi (teks kosong: {n_empty})")

lengths_df = pd.DataFrame.from_records(records)
empty_df = pd.DataFrame.from_records(empty_report)
lengths_df.head()


In [ ]:
def summarize(group: pd.DataFrame) -> pd.Series:
    lengths = group["token_length"].values
    n = len(lengths)
    n_trunc = int((lengths > MAX_LENGTH).sum())
    return pd.Series({
        "n_instances": n,
        "mean_len": np.mean(lengths),
        "median_len": np.median(lengths),
        "std_len": np.std(lengths, ddof=1) if n > 1 else 0.0,
        "min_len": int(np.min(lengths)),
        "max_len": int(np.max(lengths)),
        "n_truncated_gt_%d" % MAX_LENGTH: n_trunc,
        "pct_truncated_gt_%d" % MAX_LENGTH: 100.0 * n_trunc / n if n > 0 else np.nan,
    })


stats_df = (
    lengths_df
    .groupby(["representation", "model"], sort=False)
    .apply(summarize)
    .reset_index()
)

# Urutkan supaya enak dibaca: per representasi, lalu per model
repr_order = list(REPRESENTATIONS.keys())
model_order = list(MODEL_CHECKPOINTS.keys())
stats_df["representation"] = pd.Categorical(stats_df["representation"], categories=repr_order, ordered=True)
stats_df["model"] = pd.Categorical(stats_df["model"], categories=model_order, ordered=True)
stats_df = stats_df.sort_values(["representation", "model"]).reset_index(drop=True)

stats_df_display = stats_df.copy()
for c in ["mean_len", "median_len", "std_len"]:
    stats_df_display[c] = stats_df_display[c].round(2)
stats_df_display["pct_truncated_gt_%d" % MAX_LENGTH] = stats_df_display["pct_truncated_gt_%d" % MAX_LENGTH].round(2)

stats_df_display


In [ ]:
#sanity check
print("=== Cek jumlah instance per kombinasi ===")
expected_n = len(df)
check = stats_df[["representation", "model", "n_instances"]]
for _, row in check.iterrows():
    status = "OK" if row["n_instances"] == expected_n else "MISMATCH"
    print(f"[{status}] {row['representation']:14s} | {row['model']:9s} "
          f"-> {row['n_instances']} (expected {expected_n})")

print("\n=== Cek panjang token tidak valid (<=0) ===")
invalid = lengths_df[lengths_df["token_length"] <= 0]
print(f"Jumlah baris dengan token_length <= 0: {len(invalid)}")
if len(invalid):
    display(invalid.head(20))

print("\n=== Laporan teks kosong per (dataset, representasi, model) ===")
display(empty_df[empty_df["n_empty_text"] > 0])

In [ ]:
stats_path = os.path.join(OUTPUT_DIR, "token_length_stats.csv")
lengths_path = os.path.join(OUTPUT_DIR, "token_lengths_raw.csv")
empty_path = os.path.join(OUTPUT_DIR, "empty_text_report.csv")

stats_df_display.to_csv(stats_path, index=False)
lengths_df.to_csv(lengths_path, index=False)
empty_df.to_csv(empty_path, index=False)

print("Tersimpan:")
print(" -", stats_path)
print(" -", lengths_path)
print(" -", empty_path)


In [ ]:
lengths_df.head()

In [ ]:
# Palet kategorikal tetap (urutan tidak diputar/di-cycle), 1 warna = 1 model
MODEL_COLORS = {
    "BERT":     "#2a78d6",  # biru
    "mBERT":    "#eb6834",  # oranye
    "IndoBERT": "#1baf7a",  # aqua
}
# fallback kalau nama model beda dari default:
_fallback_palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
for i, m in enumerate(model_order):
    if m not in MODEL_COLORS:
        MODEL_COLORS[m] = _fallback_palette[i % len(_fallback_palette)]

GRID_COLOR = "#d9d9d9"
TEXT_COLOR = "#333333"


def plot_length_distributions(lengths_dfsave_path=None):
    sub = lengths_df
    reprs = [r for r in repr_order if r in sub["representation"].unique()]
    n = len(reprs)
    fig, axes = plt.subplots(n, 1, figsize=(9, 3.0 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, repr_name in zip(axes, reprs):
        for model_name in model_order:
            vals = sub[(sub["representation"] == repr_name) & (sub["model"] == model_name)]["token_length"]
            if len(vals) == 0:
                continue
            ax.hist(vals, bins=40, alpha=0.55, label=model_name,
                    color=MODEL_COLORS[model_name], edgecolor="none")
        ax.axvline(MAX_LENGTH, color="#e34948", linestyle="--", linewidth=1.5,
                    label=f"max_length={MAX_LENGTH}")
        ax.set_title(repr_name, loc="left", color=TEXT_COLOR, fontsize=11)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color(GRID_COLOR)
        ax.tick_params(colors=TEXT_COLOR)
        ax.grid(axis="y", color=GRID_COLOR, linewidth=0.6, alpha=0.6)

    axes[-1].set_xlabel("Panjang token (jumlah subword, termasuk [CLS]/[SEP])", color=TEXT_COLOR)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
               ncol=len(labels), frameon=False)
    fig.suptitle(f"Distribusi panjang token", y=1.05, color=TEXT_COLOR)
    fig.tight_layout()
    # if save_path:
    #     fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_length_distributions(lengths_df)


In [ ]:
def plot_truncation_pct(stats_df):
    sub = stats_df
    reprs = [r for r in repr_order if r in sub["representation"].unique()]
    x = np.arange(len(reprs))
    width = 0.8 / max(len(model_order), 1)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, model_name in enumerate(model_order):
        vals = []
        for r in reprs:
            row = sub[(sub["representation"] == r) & (sub["model"] == model_name)]
            pct_col = "pct_truncated_gt_%d" % MAX_LENGTH
            vals.append(row[pct_col].values[0] if len(row) else 0.0)
        bars = ax.bar(x + i * width, vals, width=width, label=model_name,
                       color=MODEL_COLORS[model_name], edgecolor="none")
        # for b, v in zip(bars, vals):
        #     if v > 0:
        #         ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%",
        #                 ha="center", va="bottom", fontsize=8, color=TEXT_COLOR)

    ax.set_xticks(x + width * (len(model_order) - 1) / 2)
    ax.set_xticklabels(reprs)
    ax.set_ylabel(f"% instance dengan panjang token > {MAX_LENGTH}", color=TEXT_COLOR)
    ax.set_title(f"Persentase instance yang akan terpotong (max_length={MAX_LENGTH})",
                 loc="left", color=TEXT_COLOR)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID_COLOR)
    ax.tick_params(colors=TEXT_COLOR)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.6, alpha=0.6)
    ax.legend(frameon=False, ncol=len(model_order))
    fig.tight_layout()
    # if save_path:
    #     fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_truncation_pct(stats_df)


In [ ]:
sub = stats_df
reprs = [r for r in repr_order if r in sub["representation"].unique()]
xlabels = [r.capitalize() for r in repr_order if r in sub["representation"].unique()]
x = np.arange(len(reprs))
width = 0.8 / max(len(model_order), 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, model_name in enumerate(model_order):
  vals = []
  for r in reprs:
      row = sub[(sub["representation"] == r) & (sub["model"] == model_name)]
      pct_col = "max_len"
      vals.append(row[pct_col].values[0])
  bars = ax.bar(x + i * width, vals, width=width, label=model_name,
                  color=MODEL_COLORS[model_name], edgecolor="black", linewidth=0.4)
  for rect, val in zip(bars, vals):
    ax.text(rect.get_x() + rect.get_width() / 2, val + 0.003, f"{val:.2f}", ha='center', va='bottom', fontsize=8)
ax.set_xticks(x + width * (len(model_order) - 1) / 2)
ax.set_xticklabels(xlabels)
ax.set_ylabel(f"Max Token Count After Tokenization", color=TEXT_COLOR)
ax.set_ylabel(f"Language Agumentations", color=TEXT_COLOR)
ax.set_title(f"Comparison of Max Token Count of Statements after Tokenization",
              color=TEXT_COLOR)


# if save_path:
#     fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
[r.capitalize() for r in repr_order if r in sub["representation"].unique()]

In [ ]:
print("=== Ringkasan statistik (semua dataset) ===")
display(stats_df_display)

print("\n=== Kombinasi dengan persentase truncation tertinggi ===")
pct_col = "pct_truncated_gt_%d" % MAX_LENGTH
display(stats_df_display.sort_values(pct_col, ascending=False).head(10))

print("\n=== Percentile panjang token (95th & 99th) per kombinasi ===")
pct_records = []
for (repr_name, model_name), g in lengths_df.groupby(["representation", "model"], sort=False):
    pct_records.append({
        "representation": repr_name,
        "model": model_name,
        "p95_len": np.percentile(g["token_length"], 95),
        "p99_len": np.percentile(g["token_length"], 99),
    })
pct_df = pd.DataFrame(pct_records)
pct_df["representation"] = pd.Categorical(pct_df["representation"], categories=repr_order, ordered=True)
pct_df["model"] = pd.Categorical(pct_df["model"], categories=model_order, ordered=True)
pct_df = pct_df.sort_values(["representation", "model"]).reset_index(drop=True)
display(pct_df)

# pct_out = os.path.join(OUTPUT_DIR, "token_length_percentiles.csv")
# pct_df.to_csv(pct_out, index=False)
# print(f"\nTersimpan: {pct_out}")


# Perbedaan Tokenize kalimat dengan LID Tag Concatenated dengan Non Tag

In [ ]:
kalimat_asli = df.loc[0].text
kalimat_dengan_label_gabung = df.loc[0].tagged_text

print("=== Pendekatan konkatenasi string (\"baik_ID\") ===")
enc_bad = tokenizer(kalimat_dengan_label_gabung, is_split_into_words=True)
bad_ids = enc_bad["input_ids"]
print(tokenizer.convert_ids_to_tokens(bad_ids))
print(f"Jumlah subword: {len(bad_ids)}")

print("\n=== Pendekatan language embedding (teks asli tidak diubah) ===")
enc_good = tokenizer(kalimat_asli, is_split_into_words=True)
good_ids = enc_good["input_ids"]
print(tokenizer.convert_ids_to_tokens(good_ids))
print(f"Jumlah subword: {len(good_ids)}")

# Perbandingan LID Naive dengan Gold Label



In [ ]:
# Sanity check cepat pada beberapa token contoh dari Tabel I di paper Anda
demo_tokens = ["interface", "friendly", "fitur", "lengkap", "update",
               "sya", "tpi", "gk", "bsa", "admin", "please", "help"]
for tok in demo_tokens:
    print(f"{tok:12s} -> {detect_word_language(tok)}")

In [ ]:
#get gold labels
#original dataset link: https://github.com/seelenbrecher/code-mixed-normalization/blob/master/data/825_Indonesian_English_CodeMixed.csv
df_gold_labels = pd.read_csv("825_Indonesian_English_CodeMixed.csv")
print("Shape:", df_gold_labels.shape)
df_gold_labels.head(3)

In [ ]:
#parsing token
def parse_tokens(x: str):
    """
    Kolom 'tokens' tersimpan sebagai representasi numpy object-array
    (format: ['raw_tweet' \n list([...])]) sehingga tidak bisa langsung
    di-parse dengan ast.literal_eval. Regex di bawah mengekstrak list token
    aktual (elemen kedua array tsb).
    """
    m = re.search(r"list\((\[.*\])\)\s*\]\s*$", x, re.S)
    if not m:
        return None
    return ast.literal_eval(m.group(1))


def parse_langs(x: str):
    return ast.literal_eval(x)


df_gold_labels["tokens_parsed"] = df_gold_labels["tokens"].apply(parse_tokens)
df_gold_labels["langs_parsed"] = df_gold_labels["langs"].apply(parse_langs)

# Validasi alignment
mismatch = (df_gold_labels["tokens_parsed"].apply(len) != df_gold_labels["langs_parsed"].apply(len)).sum()
print(f"Baris dengan panjang tokens != langs: {mismatch} / {len(df_gold_labels)}")

all_labels = Counter(l for langs in df_gold_labels["langs_parsed"] for l in langs)
print("Distribusi label gold standard:", dict(all_labels))


In [ ]:
#preprocessing token terlebih dahulu

PUNCT_RE = re.compile(r"^[\W_]+$", re.UNICODE)  # token yang hanya berisi simbol/tanda baca
URL_RE = re.compile(r"^https?://", re.IGNORECASE)

records = []  # (tweet_id, token, gold_label)

for i, row in df_gold_labels.iterrows():
    toks = row["tokens_parsed"]
    langs = row["langs_parsed"]
    for tok, gold in zip(toks, langs):
        if gold == "un":
            continue  # keluar cakupan skema ID/EN pipeline Anda
        if URL_RE.match(tok):
            continue
        if PUNCT_RE.match(tok):
            continue
        records.append((row["no"], tok, gold.upper()))

eval_df_gold_labels = pd.DataFrame(records, columns=["tweet_id", "token", "gold_label"])
eval_df_gold_labels["token_lower"] = eval_df_gold_labels["token"].str.lower()

print(f"Total token dievaluasi (setelah exclude 'un' & simbol/URL): {len(eval_df_gold_labels):,}")
print(eval_df_gold_labels["gold_label"].value_counts())
eval_df_gold_labels.head(10)

In [ ]:
#Jalankan Pipeline LID pada Token Evaluasi
# Cache prediksi per token unik supaya lebih cepat (lingua lumayan berat dipanggil per-token)
unique_tokens = eval_df_gold_labels["token_lower"].unique()
print(f"Token unik yang perlu diprediksi: {len(unique_tokens):,}")

pred_cache = {}
for tok in unique_tokens:
    pred_cache[tok] = detect_word_language(tok)

eval_df_gold_labels["pred_label"] = eval_df_gold_labels["token_lower"].map(pred_cache)

print(eval_df_gold_labels["pred_label"].value_counts())

In [ ]:
# Token yang diprediksi 'UNK' (lingua pun tidak yakin) dikeluarkan dari perhitungan
# Accuracy/F1 utama, tapi dilaporkan terpisah sebagai bagian dari analisis kegagalan.
n_unk = (eval_df_gold_labels["pred_label"] == "UNK").sum()
print(f"Token dengan prediksi UNK (lingua tidak yakin): {n_unk} "
      f"({n_unk/len(eval_df_gold_labels)*100:.2f}% dari total token evaluasi)")

scored_df_gold_labels = eval_df_gold_labels[eval_df_gold_labels["pred_label"] != "UNK"].copy()
print(f"Token yang masuk perhitungan metrik: {len(scored_df_gold_labels):,}")

In [ ]:
#Accuracy & F1-Score terhadap Gold Standard
y_true_gold_label = scored_df_gold_labels["gold_label"]
y_pred_gold_label = scored_df_gold_labels["pred_label"]

acc = accuracy_score(y_true_gold_label, y_pred_gold_label)
precision, recall, f1, support = precision_recall_fscore_support(
    y_true_gold_label, y_pred_gold_label, labels=["ID", "EN"], zero_division=0
)

print(f"Overall Accuracy: {acc:.4f}\n")
summary = pd.DataFrame({
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Support": support,
}, index=["ID", "EN"])
summary


In [ ]:
print(classification_report(y_true_gold_label, y_pred_gold_label, labels=["ID", "EN"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_true_gold_label, y_pred_gold_label, labels=["ID", "EN"])
cm_df = pd.DataFrame(cm, index=["Gold: ID", "Gold: EN"], columns=["Pred: ID", "Pred: EN"])

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: ID", "Pred: EN"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Gold: ID", "Gold: EN"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12)
ax.set_title("Confusion Matrix — LID Naif vs Gold Standard")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
#plt.savefig("confusion_matrix_lid_validation.png", dpi=150)
plt.show()

cm_df

In [ ]:
#Sample Errors
errors_df = scored_df_gold_labels[scored_df_gold_labels["gold_label"] != scored_df_gold_labels["pred_label"]].copy()
print(f"Total kesalahan: {len(errors_df):,} dari {len(scored_df_gold_labels):,} token "
      f"({len(errors_df)/len(scored_df_gold_labels)*100:.2f}%)")

error_counts = (
    errors_df.groupby("token")
    .size()
    .sort_values(ascending=False)
    .head(30)
)
error_counts.to_frame("frekuensi_salah")

In [ ]:
sample_errors = errors_df.sample(min(20, len(errors_df)), random_state=RANDOM_SEED)[
    ["token", "gold_label", "pred_label"]
].reset_index(drop=True)
sample_errors


In [ ]:
# Token yang jatuh ke fallback lingua (tidak ditemukan di kedua kamus) — berapa
# proporsinya, dan seberapa akurat lingua khusus untuk subset ini?
in_dict = scored_df_gold_labels["token_lower"].isin(english_vocab | indonesian_vocab)
fallback_df = scored_df_gold_labels[~in_dict]

if len(fallback_df) > 0:
    fb_acc = accuracy_score(fallback_df["gold_label"], fallback_df["pred_label"])
    print(f"Proporsi token yang jatuh ke fallback lingua: "
          f"{len(fallback_df)/len(scored_df_gold_labels)*100:.2f}%")
    print(f"Akurasi khusus subset fallback (lingua-only): {fb_acc:.4f}")
else:
    print("Tidak ada token yang jatuh ke fallback lingua pada sampel ini.")

In [ ]:
print("=" * 60)
print("RINGKASAN VALIDASI LID vs GOLD STANDARD (Barik et al., 2019)")
print("=" * 60)
print(f"Dataset gold standard      : {len(df_gold_labels)} tweet, "
      f"{sum(len(l) for l in df_gold_labels['langs_parsed']):,} token mentah")
print(f"Token dievaluasi (excl. un): {len(eval_df_gold_labels):,}")
print(f"Token UNK (lingua gagal)   : {n_unk} ({n_unk/len(eval_df_gold_labels)*100:.2f}%)")
print(f"Token discored              : {len(scored_df_gold_labels):,}")
print("-" * 60)
print(f"Overall Accuracy           : {acc:.4f}")
print(f"F1-Score (ID)               : {f1[0]:.4f}")
print(f"F1-Score (EN)               : {f1[1]:.4f}")
print(f"Macro F1                    : {np.mean(f1):.4f}")
print("=" * 60)

# Simpan hasil untuk dilampirkan ke paper
results_summary = pd.DataFrame({
    "metric": ["accuracy", "f1_id", "f1_en", "macro_f1", "n_evaluated", "n_unk_pct"],
    "value": [acc, f1[0], f1[1], np.mean(f1), len(scored_df_gold_labels), n_unk/len(eval_df_gold_labels)*100],
})
#results_summary.to_csv("lid_validation_summary.csv", index=False)
results_summary

# Testing LID Module ke Dataset Utama

In [ ]:
# ============================================================
# CONFIG — sesuaikan dengan dataset & setup Anda
# ============================================================

CONFIG = {
    # --- Path & format dataset Mendeley ---
    "dataset_path": "code_switched_en_id_preprocessed_filtered.csv",   # TODO: ganti dengan path file dataset Mendeley Anda

    # --- Nama kolom pada dataset (sesuaikan) ---
    # Dataset diasumsikan 1 baris = 1 kalimat (bukan 1 baris = 1 token).
    "col_text": "text",   # TODO: kolom berisi teks kalimat penuh (mis. "kalimat", "text", "tweet", dst.)
    "col_label": "sentiment",    # kolom label kategori ASLI dataset (mis. sentimen positif/negatif).
                              # Kolom ini HANYA dipakai untuk stratifikasi sampel supaya representatif,
                              # BUKAN label bahasa EN/ID -- label EN/ID selalu berasal dari anotasi manual.

    # --- Target ukuran sampel ---
    "target_tokens": 400,     # target total token (rentang 300-500 sesuai instruksi)
    "target_sentences": None, # None -> dihitung otomatis dari target_tokens & panjang rata-rata kalimat
    "random_seed": 42,

    # --- Output paths ---
    "out_dir": "lid_validation_sample",
}

Path(CONFIG["out_dir"]).mkdir(parents=True, exist_ok=True)
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
CONFIG

## Muat Dataset Mendeley & Tokenisasi

In [ ]:
def tokenize_sentence(text):
    """Tokenizer sederhana berbasis regex: kata (termasuk apostrof/tanda hubung
    di dalam kata) dan tanda baca masing-masing jadi token terpisah.

    TODO: ganti dengan tokenizer yang SAMA dengan yang dipakai korpus Barik et al.
    kalau ingin hasil benar-benar apel-ke-apel (mis. NLTK word_tokenize, tokenizer
    khusus code-mixing, dsb.).
    """
    text = "" if pd.isna(text) else str(text)
    return re.findall(r"[\w']+|[^\w\s]", text, flags=re.UNICODE)


def load_mendeley_dataset(path):
    """Muat dataset Mendeley (1 baris = 1 kalimat) lalu tokenisasi jadi token-level.

    Return:
        df_tokens: dataframe token-level (sentence_id, token_pos, token, sentence_label)
        df_sentences: dataframe kalimat-level (sentence_id, sentence_text, sentence_label)
    """
    raw = pd.read_csv(path)

    required = [CONFIG["col_text"], CONFIG["col_label"]]
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise ValueError(
            f"Kolom berikut tidak ditemukan di dataset: {missing}. "
            f"Kolom yang tersedia: {list(raw.columns)}. "
            "Sesuaikan CONFIG['col_text']/['col_label']."
        )

    raw = raw.rename(columns={
        CONFIG["col_text"]: "sentence_text",
        CONFIG["col_label"]: "sentence_label",
    })[["sentence_text", "sentence_label"]].reset_index(drop=True)
    raw["sentence_id"] = raw.index

    token_rows = []
    for _, r in raw.iterrows():
        for pos, tok in enumerate(tokenize_sentence(r["sentence_text"])):
            token_rows.append({
                "sentence_id": r["sentence_id"],
                "token_pos": pos,
                "token": tok,
                "sentence_label": r["sentence_label"],
            })

    df_tokens = pd.DataFrame(token_rows)
    df_sentences = raw[["sentence_id", "sentence_text", "sentence_label"]]
    return df_tokens, df_sentences


df_tokens, df_sentences = load_mendeley_dataset(CONFIG["dataset_path"])
print(f"Total kalimat: {len(df_sentences)} | Total token hasil tokenisasi: {len(df_tokens)}")
df_tokens.head(10)

## Stratified Sampling

In [ ]:
sent_stats = (
    df_tokens.groupby("sentence_id")
    .agg(n_tokens=("token", "size"))
    .reset_index()
    .merge(df_sentences[["sentence_id", "sentence_label"]], on="sentence_id")
)

avg_sent_len = sent_stats["n_tokens"].mean()
print(f"Panjang kalimat rata-rata: {avg_sent_len:.2f} token")
print(sent_stats["sentence_label"].value_counts())

# hitung target jumlah kalimat dari target_tokens, di-cap ke [50, 80]
if CONFIG["target_sentences"] is None:
    n_sentences_target = int(round(CONFIG["target_tokens"] / avg_sent_len))
    n_sentences_target = max(50, min(80, n_sentences_target))
else:
    n_sentences_target = CONFIG["target_sentences"]

print(f"Target jumlah kalimat sampel: {n_sentences_target}")


In [ ]:
def stratified_sentence_sample(sent_stats, n_target, seed):
    """Ambil sampel kalimat acak, berimbang antar label kategori dataset."""
    classes = sent_stats["sentence_label"].unique().tolist()
    n_per_class = n_target // len(classes)

    parts = []
    rng = np.random.RandomState(seed)
    for cls in classes:
        pool = sent_stats[sent_stats["sentence_label"] == cls]
        n_take = min(n_per_class, len(pool))
        parts.append(pool.sample(n=n_take, random_state=rng))

    sample = pd.concat(parts).sample(frac=1.0, random_state=rng)  # acak urutan akhir
    return sample.reset_index(drop=True)


sampled_sentences = stratified_sentence_sample(sent_stats, n_sentences_target, CONFIG["random_seed"])
print(sampled_sentences["sentence_label"].value_counts())

sample_df = (
    df_tokens[df_tokens["sentence_id"].isin(sampled_sentences["sentence_id"])]
    .sort_values(["sentence_id", "token_pos"])
    .reset_index(drop=True)
)

print(f"Total kalimat tersampel: {sample_df['sentence_id'].nunique()}")
print(f"Total token tersampel: {len(sample_df)}")
print(sample_df["sentence_label"].value_counts())

sample_path = Path(CONFIG["out_dir"]) / "sampled_tokens_master.csv"
sample_df.to_csv(sample_path, index=False)
print(f"Sampel master (untuk keperluan internal/pengecekan) disimpan di: {sample_path}")

sampled_sentences_context_path = Path(CONFIG["out_dir"]) / "sampled_sentences_context.csv"
df_sentences[df_sentences["sentence_id"].isin(sampled_sentences["sentence_id"])].to_csv(
    sampled_sentences_context_path, index=False
)
print(f"Teks kalimat penuh (konteks untuk annotator) disimpan di: {sampled_sentences_context_path}")

sample_df.head(15)


In [ ]:
# new cell after cell 125
pool = sent_stats[~sent_stats.sentence_id.isin(sample_df.sentence_id.unique())]
new = stratified_sentence_sample(pool, 100, seed=2025)

## Export Blind Annotation Sample

In [ ]:
def build_annotation_template(sample_df):
    tmpl = sample_df[["sentence_id", "token_pos", "token"]].copy()
    tmpl["label"] = ""  # <-- diisi manual: "EN" atau "ID"
    return tmpl


annotation_template = build_annotation_template(sample_df)

annot1_path = Path(CONFIG["out_dir"]) / "annotator_1_TO_FILL.csv"
annotation_template.to_csv(annot1_path, index=False)
print(f"Template annotator 1: {annot1_path}")

annotation_template.head(10)

In [ ]:
sample_df2 = (df_tokens[df_tokens.sentence_id.isin(new.sentence_id)]
              .sort_values(["sentence_id", "token_pos"]).reset_index(drop=True))
build_annotation_template(sample_df2).to_csv(
    Path(CONFIG["out_dir"]) / "annotator_1_batch2_TO_FILL.csv", index=False)
print(sample_df2.sentence_id.nunique(), len(sample_df2))

In [ ]:
final_sample_df = pd.concat([sample_df, sample_df2])
final_sample_df.shape

In [ ]:
#CONCATENATE BOTH CSV
annot_batch1 = pd.read_csv(annot1_path)
annot_batch2 = pd.read_csv(Path(CONFIG["out_dir"]) / "annotator_1_batch2_TO_FILL.csv")

In [ ]:
concatenated_batch = pd.concat([annot_batch1, annot_batch2])
concatenated_batch.to_csv(Path(CONFIG["out_dir"]) / "annotator_batch_combined_TO_FILL.csv")

## Muat Kembali Hasil Anotasi

In [ ]:
def load_annotations(path):
    df = pd.read_csv(path)
    df["label"] = df["label"].astype(str).str.strip().str.upper()
    invalid = ~df["label"].isin(["EN", "ID"])
    if invalid.any():
        print(f"[WARNING] {invalid.sum()} baris di {path} berisi label tidak valid (bukan EN/ID):")
        print(df.loc[invalid, ["sentence_id", "token_pos", "token", "label"]])
    return df


# TODO: pastikan file ini sudah diisi manual sebelum dijalankan
annot1 = load_annotations(Path(CONFIG["out_dir"]) / "annotator_batch_combined_TO_FILL.csv")


## Susun Gold Label

In [ ]:
gold_df = annot1.rename(columns={"label": "gold_label"})[
        ["sentence_id", "token_pos", "token", "gold_label"]
    ]

gold_path = Path(CONFIG["out_dir"]) / "gold_labels.csv"
gold_df.to_csv(gold_path, index=False)
print(f"Gold labels disimpan di: {gold_path}")
gold_df.head(10)

In [ ]:
gold_df.gold_label.value_counts()

## Jalankan Modul LID pada Sampel

In [ ]:
def run_lid_module(token):
    return detect_word_language(token)

In [ ]:
predictions = []
for _, row in gold_df.iterrows():
    pred = run_lid_module(row["token"])
    predictions.append(pred)

eval_df = gold_df.copy()
eval_df["pred_label"] = predictions
eval_df["pred_label"] = eval_df["pred_label"].astype(str).str.strip().str.upper()

eval_path = Path(CONFIG["out_dir"]) / "lid_predictions_vs_gold.csv"
eval_df.to_csv(eval_path, index=False)
print(f"Prediksi LID vs gold disimpan di: {eval_path}")
eval_df.head(10)


## Precision / Recall / F1 / Macro F1

In [ ]:
labels_order = ["EN", "ID"]

y_true = eval_df["gold_label"]
y_pred = eval_df["pred_label"]

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_order, zero_division=0
)

table1 = pd.DataFrame({
    "Label": labels_order,
    "Precision": np.round(precision, 4),
    "Recall": np.round(recall, 4),
    "F1-Score": np.round(f1, 4),
    "Support": support,
})

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_order, average="macro", zero_division=0
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_order, average="weighted", zero_division=0
)

summary_rows = pd.DataFrame([
    {"Label": "Macro Avg", "Precision": round(macro_p, 4), "Recall": round(macro_r, 4),
     "F1-Score": round(macro_f1, 4), "Support": support.sum()},
    {"Label": "Weighted Avg", "Precision": round(weighted_p, 4), "Recall": round(weighted_r, 4),
     "F1-Score": round(weighted_f1, 4), "Support": support.sum()},
])

table1_final = pd.concat([table1, summary_rows], ignore_index=True)

table1_out_path = Path(CONFIG["out_dir"]) / "table1_validation_results.csv"
table1_final.to_csv(table1_out_path, index=False)
print(f"Tabel hasil disimpan di: {table1_out_path}\n")
table1_final


In [ ]:
print("Confusion matrix (baris = gold, kolom = prediksi):")
cm = confusion_matrix(y_true, y_pred, labels=labels_order)
pd.DataFrame(cm, index=[f"gold_{l}" for l in labels_order], columns=[f"pred_{l}" for l in labels_order])


In [ ]:
print(classification_report(y_true, y_pred, labels=labels_order, zero_division=0))


In [ ]:
print(f"Total kalimat tersampel : {final_sample_df['sentence_id'].nunique()}")
print(f"Total token tersampel   : {len(final_sample_df)}")
print(f"Macro F1 (sampel ini)   : {macro_f1:.4f}")


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

ev = pd.read_csv("lid_validation_sample/lid_predictions_vs_gold.csv")
groups = [g for _, g in ev.groupby("sentence_id")]
rng = np.random.default_rng(42)
def mf1(df): return f1_score(df.gold_label, df.pred_label, labels=["EN","ID"], average="macro", zero_division=0)

boot = [mf1(pd.concat([groups[i] for i in rng.integers(0, len(groups), len(groups))]))
        for _ in range(5000)]
print(len(ev), mf1(ev), np.percentile(boot, [2.5, 97.5]))